# 表現力を高めたGaussian Process

このNotebookでは `JointEncoderGP` (DKL)、`InfiniteWidthBNNGP`、`SpectralMixtureGP`、`SingleTaskDeepGP` の位置づけと基本APIを比較します。モデルごとに表現力の源と学習方法が異なる点を確認します。

## このNotebookで学ぶこと

- DKL・Infinite-width BNN GP・Spectral Mixture GP・DeepGPの**表現力の源**を区別する
- Exact MLLとJoint/Variational学習の違いを理解する
- 学習後のPosterior（平均と潜在関数の不確実性）を比較する
- 事後平均の有限候補選択と、**qLogEIを用いた獲得関数ベースの候補提案**を比較する
- Mixed・MultiTask・Kroneckerでは、カテゴリやタスクの入力・出力構造を意識する

このNotebookは軽量な教育用設定です。少ない学習epochでの精度やモデル間の優劣を保証するものではありません。

In [ ]:
import torch
from robotorchan.models import InfiniteWidthBNNGP, JointEncoderGP, SpectralMixtureGP
from robotorchan.models.expressive.deep_gp import SingleTaskDeepGP

torch.manual_seed(0)
train_X = torch.rand(12, 2, dtype=torch.double)
train_Y = torch.sin(2 * torch.pi * train_X[:, :1]) + 0.2 * train_X[:, 1:2]


## Exact GPとして扱える表現力モデル

DKLは入力表現をニューラルネットワークで学習します。Infinite-width BNN GPはニューラルネットワーク由来の解析kernelを使い、Spectral Mixture GPは周波数構造をkernelで表現します。3モデルともExact GPのposterior contractを維持します。

### 表現力の源はそれぞれ異なる

| モデル | 何を柔軟にするか | 主な注意点 |
|---|---|---|
| DKL (`JointEncoderGP`) | NNで学習する入力特徴表現 | EncoderとGPを同時学習する |
| Infinite-width BNN GP | NN由来の解析的な共分散関数 | 無限幅近似のカーネルを使うExact GP |
| Spectral Mixture GP | 複数の周波数成分を表すカーネル | 周期・準周期構造に適する |
| DeepGP | 確率的なGP層の合成 | 変分推論とPosterior近似を使う |

**重要：** DKLとDeepGPは同じものではありません。DKLのEncoderは通常決定論的な特徴写像ですが、DeepGPは中間層も確率的です。

In [ ]:
models = {
    "DKL": JointEncoderGP(train_X, train_Y, latent_dim=2, hidden_dims=(6,), random_state=0),
    "I-BNN": InfiniteWidthBNNGP(train_X, train_Y, depth=2),
    "SM": SpectralMixtureGP(train_X, train_Y, num_mixtures=2),
}

test_X = torch.linspace(0, 1, 8, dtype=torch.double)
test_X = torch.stack([test_X, torch.full_like(test_X, 0.5)], dim=-1)
for name, model in models.items():
    model.eval()
    model.likelihood.eval()
    posterior = model.posterior(test_X)
    print(name, posterior.mean.shape, posterior.variance.shape)


## Deep Gaussian Process

DeepGPは中間表現も確率変数です。そのためExact GPとは学習objectiveとposterior samplingが異なります。ここでは構築とposterior contractを確認し、重いtraining loopはbenchmarkや専用テストに任せます。

In [ ]:
deep_gp = SingleTaskDeepGP(
    train_X,
    train_Y,
    hidden_dims=(3,),
    num_inducing=5,
    posterior_samples=16,
    random_state=0,
)
deep_gp.eval()
posterior = deep_gp.posterior(test_X)
print("DeepGP", posterior.mean.shape, posterior.variance.shape)


## 使い分け

- **DKL**: 目的に有用な非線形特徴をデータから学習したい場合
- **DeepGP**: 階層的で確率的な潜在表現を必要とする場合
- **Infinite-width BNN GP**: neural-network由来priorとExact GP推論を両立したい場合
- **Spectral Mixture GP**: 周期・準周期・複数周波数を持つ定常構造を表現したい場合

表現力が高いモデルを自動的に選ぶのではなく、標準 `SingleTaskGP` と予測精度、不確実性校正、計算時間、BO性能を比較して選択します。

## 学習方法の違い

DKLはencoderとGPを共同学習します。I-BNNとSpectral MixtureはExact MLL、DeepGPは変分ELBOを使用します。Joint NeuralとDeepGPはそれぞれの損失を用いた`fit()`を利用します。


### `fit()` と `fit_gpytorch_mll()` の違い

- I-BNN・Spectral Mixture：`make_mll()` でExact MLLを作り、BoTorchの `fit_gpytorch_mll()` で最適化
- DKL：`model.fit(epochs=..., lr=...)` によりEncoderとGPを共同学習
- DeepGP：`model.fit(epochs=..., lr=...)` により変分学習

構築直後の `posterior()` はAPI確認用です。**探索に使う予測は学習後のPosterior**を参照します。

In [ ]:
from botorch.fit import fit_gpytorch_mll

for name in ("I-BNN", "SM"):
    model = models[name]
    fit_gpytorch_mll(model.make_mll())
    model.eval()

for model in (models["DKL"], deep_gp):
    model.fit(epochs=3, lr=0.01)
    print(type(model).__name__, "fitted")


### 学習済みモデルのPosteriorを可視化

`x1=0.5` に固定し、`x0` を0〜1で動かした断面を比較します。帯は潜在関数の事後標準偏差の約1.96倍です。DeepGPのPosteriorはサンプリングによる近似であり、同じ形の帯でもExact GPと推定の性質が異なります。

In [ ]:
import matplotlib.pyplot as plt

plot_x0 = torch.linspace(0, 1, 101, dtype=train_X.dtype)
plot_X = torch.stack([plot_x0, torch.full_like(plot_x0, 0.5)], dim=-1)
true_y = torch.sin(2 * torch.pi * plot_x0) + 0.1
fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True, sharey=True)
for ax, (name, surrogate) in zip(axes.flat, {**models, "DeepGP": deep_gp}.items()):
    surrogate.eval()
    with torch.no_grad():
        post = surrogate.posterior(plot_X)
        mean = post.mean.reshape(-1)
        std = post.variance.clamp_min(0).sqrt().reshape(-1)
    ax.plot(plot_x0, true_y, "--", label="true slice")
    ax.plot(plot_x0, mean, label="posterior mean")
    ax.fill_between(plot_x0, mean - 1.96 * std,
                    mean + 1.96 * std, alpha=0.2,
                    label="approx. 95% latent interval")
    ax.set_title(name)
    ax.set_xlabel("x0 (x1=0.5)")
    ax.set_ylabel("objective")
axes[0, 0].legend(fontsize=8)
plt.tight_layout()
plt.show()

## 候補提案と目的関数評価

各モデルの元の2次元入力空間で候補プールを評価し、事後平均が最大となる候補を提案します。DeepGPも同じ公開入力座標を使用します。


In [ ]:
candidate_pool = torch.rand(64, 2, dtype=torch.double)


def objective(X):
    return torch.sin(2 * torch.pi * X[..., :1]) + 0.2 * X[..., 1:2]


for name, model in {**models, "DeepGP": deep_gp}.items():
    with torch.no_grad():
        scores = model.posterior(candidate_pool).mean.squeeze(-1)
    candidate = candidate_pool[scores.argmax()].unsqueeze(0)
    assert candidate.shape == torch.Size([1, 2])
    print(name, "candidate:", candidate)
    print(name, "posterior mean:", scores.max())
    print(name, "objective:", objective(candidate))


### 事後平均による候補選択は比較例

次の既存セルは各モデルのPosterior Meanが最大となる有限候補を選びます。これは予測平均の活用（exploitation）であり、不確実性を使った改善期待値の最適化ではありません。

通常のExact GPであるI-BNNとSpectral Mixtureでは、次に**qLogEI + `optimize_acqf()`** による連続空間の候補最適化を示します。Joint/DeepGPには、既存の事後平均例を維持し、全モデルへの無条件な獲得関数適用は避けます。

### Exact GPモデルでqLogEIを最適化

`qLogExpectedImprovement` は、学習済みPosteriorの平均と不確実性に基づく改善の期待値を使います。`optimize_acqf()` は元の2次元入力空間で獲得関数を最適化します。提案後に合成目的関数を評価します。

In [ ]:
from botorch.acquisition.logei import qLogExpectedImprovement
from botorch.optim import optimize_acqf
from botorch.sampling.normal import SobolQMCNormalSampler

bounds = torch.stack([torch.zeros(2, dtype=train_X.dtype),
                      torch.ones(2, dtype=train_X.dtype)])
for name in ("I-BNN", "SM"):
    surrogate = models[name]
    acqf = qLogExpectedImprovement(
        model=surrogate,
        best_f=train_Y.max(),
        sampler=SobolQMCNormalSampler(sample_shape=torch.Size([32])),
    )
    proposed, acq_value = optimize_acqf(
        acq_function=acqf, bounds=bounds, q=1,
        num_restarts=3, raw_samples=32,
    )
    assert proposed.shape == (1, 2)
    print(name, "qLogEI candidate:", proposed)
    print("acquisition:", acq_value, "objective:", objective(proposed))

## Mixed-input exact expressive GPs

カテゴリ変数を保持したInfinite-width BNN kernelとSpectral Mixture kernelを学習し、元の入力空間で候補を選択します。有限候補の予測平均順位であり、獲得関数最適化ではありません。


In [ ]:
from robotorchan.models import MixedInfiniteWidthBNNGP, MixedSpectralMixtureGP

mixed_train_X = torch.cat(
    [train_X, torch.randint(0, 2, (train_X.shape[0], 1)).to(train_X)],
    dim=-1,
)
mixed_candidate_pool = torch.cat(
    [candidate_pool, torch.randint(0, 2, (candidate_pool.shape[0], 1)).to(candidate_pool)],
    dim=-1,
)
mixed_exact_models = {
    "MixedInfiniteWidthBNNGP": MixedInfiniteWidthBNNGP(
        mixed_train_X, train_Y, cat_dims=[2], depth=2
    ),
    "MixedSpectralMixtureGP": MixedSpectralMixtureGP(
        mixed_train_X, train_Y, cat_dims=[2], num_mixtures=2
    ),
}
for name, model in mixed_exact_models.items():
    fit_gpytorch_mll(model.make_mll())
    model.eval()
    with torch.no_grad():
        scores = model.posterior(mixed_candidate_pool).mean.reshape(-1)
    candidate = mixed_candidate_pool[scores.argmax()].unsqueeze(0)
    assert candidate.shape == (1, 3)
    assert candidate[0, -1].item() in (0, 1)
    print(name, "candidate:", candidate)
    print(name, "objective:", objective(candidate[..., :2]))


### Mixedモデルの候補制約

Mixed入力では連続2次元とカテゴリ1次元を区別します。カテゴリは有効な値（この例では0または1）だけを取り、連続値のように補間しません。

後続の有限プール例に加え、Exact Mixed GPに対してカテゴリを列挙したqLogEI最適化も確認します。

In [ ]:
from botorch.optim import optimize_acqf_mixed

mixed_bounds = torch.stack([
    torch.zeros(3, dtype=train_X.dtype),
    torch.ones(3, dtype=train_X.dtype),
])
for name, surrogate in mixed_exact_models.items():
    acqf = qLogExpectedImprovement(
        model=surrogate,
        best_f=train_Y.max(),
        sampler=SobolQMCNormalSampler(sample_shape=torch.Size([32])),
    )
    proposed, acq_value = optimize_acqf_mixed(
        acq_function=acqf, bounds=mixed_bounds,
        q=1, num_restarts=3, raw_samples=32,
        fixed_features_list=[{2: 0.0}, {2: 1.0}],
    )
    assert proposed.shape == (1, 3)
    assert proposed[0, 2].item() in (0, 1)
    print(name, "qLogEI candidate:", proposed)
    print("acquisition:", acq_value,
          "objective:", objective(proposed[..., :2]))

## Expressive long-format multitask GPs

Infinite-width BNNとSpectral Mixtureのタスク共分散モデルを学習します。タスク0の候補を有限プールから予測平均で選択します。


In [ ]:
from robotorchan.models import InfiniteWidthBNNMultiTaskGP, SpectralMixtureMultiTaskGP

multitask_train_X = torch.cat(
    [
        torch.cat([train_X, torch.zeros(train_X.shape[0], 1, dtype=train_X.dtype)], dim=-1),
        torch.cat([train_X, torch.ones(train_X.shape[0], 1, dtype=train_X.dtype)], dim=-1),
    ],
    dim=0,
)
multitask_train_Y = torch.cat([train_Y, train_Y + 0.2], dim=0)
multitask_pool = torch.cat(
    [candidate_pool, torch.zeros(candidate_pool.shape[0], 1, dtype=candidate_pool.dtype)],
    dim=-1,
)
expressive_multitask_models = {
    "InfiniteWidthBNNMultiTaskGP": InfiniteWidthBNNMultiTaskGP(
        multitask_train_X, multitask_train_Y, task_feature=2, depth=2
    ),
    "SpectralMixtureMultiTaskGP": SpectralMixtureMultiTaskGP(
        multitask_train_X, multitask_train_Y, task_feature=2, num_mixtures=2
    ),
}
for name, model in expressive_multitask_models.items():
    fit_gpytorch_mll(model.make_mll())
    model.eval()
    with torch.no_grad():
        scores = model.posterior(multitask_pool).mean.reshape(-1)
    candidate = multitask_pool[scores.argmax()].unsqueeze(0)
    assert candidate.shape == (1, 3)
    print(name, "candidate:", candidate)
    print(name, "task 0 objective:", objective(candidate[..., :2]))


### MultiTaskの探索ではタスクを固定する

long-format MultiTaskは入力の末尾にタスクIDを持ちます。事後平均による有限プール選択は比較用に残します。

獲得関数を使う場合は、対象タスクの固定と**スカラー目的関数の指定**が重要です。MultiTaskモデルは `num_outputs` が複数となることがあり、BoTorchのMC獲得関数に目的関数を渡さないとエラーになる場合があります。

## Mixed expressive long-format multitask GPs

連続2次元、カテゴリ1次元、タスク識別子1次元の入力を使用します。カテゴリとタスクを保持したまま、タスク0の候補を予測平均で選択します。


In [ ]:
from robotorchan.models import (
    MixedInfiniteWidthBNNMultiTaskGP,
    MixedSpectralMixtureMultiTaskGP,
)

mixed_multitask_train_X = torch.cat(
    [
        torch.cat(
            [mixed_train_X, torch.zeros(mixed_train_X.shape[0], 1, dtype=train_X.dtype)],
            dim=-1,
        ),
        torch.cat(
            [mixed_train_X, torch.ones(mixed_train_X.shape[0], 1, dtype=train_X.dtype)],
            dim=-1,
        ),
    ],
    dim=0,
)
mixed_multitask_pool = torch.cat(
    [
        mixed_candidate_pool,
        torch.zeros(mixed_candidate_pool.shape[0], 1, dtype=train_X.dtype),
    ],
    dim=-1,
)
mixed_expressive_multitask_models = {
    "MixedInfiniteWidthBNNMultiTaskGP": MixedInfiniteWidthBNNMultiTaskGP(
        mixed_multitask_train_X, multitask_train_Y,
        task_feature=3, cat_dims=[2], depth=2,
    ),
    "MixedSpectralMixtureMultiTaskGP": MixedSpectralMixtureMultiTaskGP(
        mixed_multitask_train_X, multitask_train_Y,
        task_feature=3, cat_dims=[2], num_mixtures=2,
    ),
}
for name, model in mixed_expressive_multitask_models.items():
    fit_gpytorch_mll(model.make_mll())
    model.eval()
    with torch.no_grad():
        scores = model.posterior(mixed_multitask_pool).mean.reshape(-1)
    candidate = mixed_multitask_pool[scores.argmax()].unsqueeze(0)
    assert candidate.shape == (1, 4)
    assert candidate[0, 2].item() in (0, 1)
    assert candidate[0, 3].item() == 0
    print(name, "candidate:", candidate)
    print(name, "task 0 objective:", objective(candidate[..., :2]))


## Expressive Kronecker multitask GPs

共通入力点で2タスクを同時観測するblock-designモデルを学習します。候補はタスク0の予測平均で選択します。


In [ ]:
from botorch.optim.fit import fit_gpytorch_mll_torch
from robotorchan.models import (
    InfiniteWidthBNNKroneckerMultiTaskGP,
    SpectralMixtureKroneckerMultiTaskGP,
)

block_train_Y = torch.cat([train_Y, train_Y + 0.2], dim=-1)
expressive_kronecker_models = {
    "InfiniteWidthBNNKroneckerMultiTaskGP": InfiniteWidthBNNKroneckerMultiTaskGP(
        train_X, block_train_Y, depth=2
    ),
    "SpectralMixtureKroneckerMultiTaskGP": SpectralMixtureKroneckerMultiTaskGP(
        train_X, block_train_Y, num_mixtures=2
    ),
}
for name, model in expressive_kronecker_models.items():
    fit_gpytorch_mll(
        model.make_mll(),
        optimizer=fit_gpytorch_mll_torch,
        optimizer_kwargs={"step_limit": 20},
    )
    model.eval()
    with torch.no_grad():
        scores = model.posterior(candidate_pool).mean[:, 0]
    candidate = candidate_pool[scores.argmax()].unsqueeze(0)
    assert candidate.shape == (1, 2)
    print(name, "candidate:", candidate)
    print(name, "task 0 objective:", objective(candidate))


### Kroneckerモデルの出力

Kronecker型は同じ入力に対する複数タスクのPosteriorを返します。後続の候補選択はタスク0の平均だけを順位付けします。qLogEIなどのMC獲得関数に拡張する場合は、対象出力をスカラー化する `objective` または `posterior_transform` が必要です。

## Mixed expressive Kronecker multitask GPs

カテゴリ入力を含む共通観測点で2タスクを学習し、タスク0の候補を有限プールから提案します。獲得関数最適化ではなく予測平均に基づく候補選択です。


In [ ]:
from robotorchan.models import (
    MixedInfiniteWidthBNNKroneckerMultiTaskGP,
    MixedSpectralMixtureKroneckerMultiTaskGP,
)

mixed_expressive_kronecker_models = {
    "MixedInfiniteWidthBNNKroneckerMultiTaskGP": (
        MixedInfiniteWidthBNNKroneckerMultiTaskGP(
            mixed_train_X, block_train_Y, cat_dims=[2], depth=2
        )
    ),
    "MixedSpectralMixtureKroneckerMultiTaskGP": (
        MixedSpectralMixtureKroneckerMultiTaskGP(
            mixed_train_X, block_train_Y, cat_dims=[2], num_mixtures=2
        )
    ),
}
for name, model in mixed_expressive_kronecker_models.items():
    fit_gpytorch_mll(
        model.make_mll(),
        optimizer=fit_gpytorch_mll_torch,
        optimizer_kwargs={"step_limit": 20},
    )
    model.eval()
    with torch.no_grad():
        scores = model.posterior(mixed_candidate_pool).mean[:, 0]
    candidate = mixed_candidate_pool[scores.argmax()].unsqueeze(0)
    assert candidate.shape == (1, 3)
    assert candidate[0, -1].item() in (0, 1)
    print(name, "candidate:", candidate)
    print(name, "task 0 objective:", objective(candidate[..., :2]))


## DeepGP variants: multitask and mixed inputs

DeepGPのLong-format MultiTask、Mixed SingleTask、Mixed MultiTaskを軽量な変分学習で確認します。候補選択は元の入力空間における有限プールの予測平均順位です。


In [ ]:
from robotorchan.models.expressive.deep_gp import (
    MixedMultiTaskDeepGP,
    MixedSingleTaskDeepGP,
    MultiTaskDeepGP,
)

deep_gp_variants = [
    (
        "MultiTaskDeepGP",
        MultiTaskDeepGP(
            multitask_train_X, multitask_train_Y, task_feature=2,
            hidden_dims=(3,), num_inducing=5, posterior_samples=16,
        ),
        multitask_pool,
    ),
    (
        "MixedSingleTaskDeepGP",
        MixedSingleTaskDeepGP(
            mixed_train_X, train_Y, cat_dims=[2],
            hidden_dims=(3,), num_inducing=5, posterior_samples=16,
        ),
        mixed_candidate_pool,
    ),
    (
        "MixedMultiTaskDeepGP",
        MixedMultiTaskDeepGP(
            mixed_multitask_train_X, multitask_train_Y,
            task_feature=3, cat_dims=[2],
            hidden_dims=(3,), num_inducing=5, posterior_samples=16,
        ),
        mixed_multitask_pool,
    ),
]
for name, model, pool in deep_gp_variants:
    model.fit(epochs=3, lr=0.01)
    with torch.no_grad():
        scores = model.posterior(pool).mean.reshape(-1)
    candidate = pool[scores.argmax()].unsqueeze(0)
    assert candidate.shape == (1, pool.shape[-1])
    print(name, "candidate:", candidate)
    print(name, "objective:", objective(candidate[..., :2]))


## まとめ：学習と候補提案の対応

| モデル群 | 学習 | このNotebookの候補提案 |
|---|---|---|
| I-BNN / Spectral Mixture | Exact MLL | 事後平均プール比較 + qLogEI連続最適化 |
| DKL / DeepGP | 各モデルの `fit()` | 事後平均プール比較 |
| Mixed I-BNN / Spectral Mixture | Exact MLL | 事後平均プール比較 + qLogEI Mixed最適化 |
| MultiTask / Mixed MultiTask | Exact MLL | タスク0の事後平均プール比較 |
| Kronecker / Mixed Kronecker | Exact MLL | 出力0の事後平均プール比較 |
| DeepGP派生モデル | 各モデルの `fit()` | 事後平均プール比較 |

全モデルで構築・学習・候補提案・合成目的関数による評価を確認します。qLogEIによる獲得関数最適化は、まず適合が明確なExact GPとMixed Exact GPに追加しています。